# Silver Transformation — TfL Line Status

Transform raw TfL line-status snapshots from Bronze into structured, validated Silver records.

This notebook:

1. Reads the Bronze Delta table.
2. Parses the raw JSON payload using an explicit schema.
3. Explodes nested line and status arrays.
4. Applies data-quality rules.
5. Removes duplicate records.
6. Merges validated rows into the Silver Delta table.

**Source:** `workspace.urbanpulse_bronze.tfl_line_status`

**Target:** `workspace.urbanpulse_silver.tfl_line_status`

## 1. Initialise project paths

Add the repository `src` directory to the Python path so shared UrbanPulse modules can be imported.

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import transformation and quality components

Transformation logic and data-quality rules are maintained outside the notebook so they can be reused and tested independently.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

from urbanpulse.transformations.tfl_line_status import (
    transform_tfl_line_status,
)

from urbanpulse.quality.tfl_line_status import (
    valid_tfl_line_status,
    invalid_tfl_line_status,
)

## 3. Define source and target tables

In [0]:
BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "tfl_line_status"
)

SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "tfl_line_status"
)

## 4. Read Bronze snapshots

Bronze contains one raw JSON payload for each successful TfL API request.

In [0]:
bronze_df = spark.table(
    BRONZE_TABLE
)

print(
    f"Bronze snapshots: "
    f"{bronze_df.count()}"
)

display(
    bronze_df.select(
        "request_id",
        "ingested_at",
        "http_status",
    )
)

## 5. Parse and flatten the TfL payload

The raw JSON array is parsed using the explicit TfL schema.

Each Tube line and its nested status record are converted into individual rows.

In [0]:
parsed_df = transform_tfl_line_status(
    bronze_df
)

print(
    f"Parsed rows: "
    f"{parsed_df.count()}"
)

display(parsed_df)

## 6. Apply data-quality rules

Rows required for downstream analysis must contain:

- request ID
- line ID
- line name
- snapshot timestamp
- status severity

Rows that fail these minimum requirements are separated from valid records.

In [0]:
valid_df = valid_tfl_line_status(
    parsed_df
)

invalid_df = invalid_tfl_line_status(
    parsed_df
)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print(
    f"Valid rows: {valid_count}"
)

print(
    f"Invalid rows: {invalid_count}"
)

## 7. Enforce the data contract

For this dataset, invalid structural records are treated as a pipeline failure.

A quarantine mechanism will be introduced later for sources where partial acceptance is appropriate.

In [0]:
if invalid_count > 0:
    display(invalid_df)

    raise ValueError(
        f"Data quality failure: "
        f"{invalid_count} invalid rows"
    )

print("Data quality checks passed.")

## 8. Remove duplicate records

A Silver record is uniquely identified by the API request, Tube line, and TfL status record.

Duplicate source records are removed before the Delta merge.

In [0]:
deduplicated_df = (
    valid_df
    .dropDuplicates([
        "request_id",
        "line_id",
        "status_id",
    ])
)

print(
    f"Rows after deduplication: "
    f"{deduplicated_df.count()}"
)

## 9. Add Silver processing metadata

`processed_at` records when the Bronze record was transformed into Silver.

In [0]:
silver_df = (
    deduplicated_df
    .withColumn(
        "processed_at",
        F.current_timestamp(),
    )
)

## 10. Merge into Silver

The first execution creates the Delta table.

Subsequent executions use Delta `MERGE` so rerunning the notebook does not duplicate previously processed snapshots.

In [0]:
if not spark.catalog.tableExists(
    SILVER_TABLE
):
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(
            SILVER_TABLE
        )
    )

    print(
        f"Created Silver table: "
        f"{SILVER_TABLE}"
    )

else:
    silver_table = DeltaTable.forName(
        spark,
        SILVER_TABLE,
    )

    (
        silver_table.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.request_id = source.request_id
            AND target.line_id = source.line_id
            AND target.status_id <=> source.status_id
            """,
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        f"Merged into Silver table: "
        f"{SILVER_TABLE}"
    )

## 11. Verify the Silver table

Inspect the resulting structured records and confirm snapshot counts.

In [0]:
%sql
SELECT
    request_id,
    line_id,
    line_name,
    status_severity,
    status_description,
    status_reason,
    snapshot_at,
    processed_at
FROM workspace.urbanpulse_silver.tfl_line_status
ORDER BY snapshot_at DESC, line_name;

In [0]:
%sql
SELECT
    COUNT(*) AS silver_rows,
    COUNT(DISTINCT request_id) AS snapshots,
    COUNT(DISTINCT line_id) AS lines
FROM workspace.urbanpulse_silver.tfl_line_status;

In [0]:
%sql
SELECT COUNT(*) AS silver_rows
FROM workspace.urbanpulse_silver.tfl_line_status;

In [0]:
%sql
SELECT
    line_name,
    status_severity,
    status_description,
    status_reason,
    snapshot_at
FROM workspace.urbanpulse_silver.tfl_line_status
WHERE status_description <> 'Good Service'
ORDER BY snapshot_at DESC;

In [0]:
%sql
DESCRIBE HISTORY
workspace.urbanpulse_silver.tfl_line_status;